[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/limorigu/uchicago-python-ai-workshop/blob/main/notebooks/01b_excel_to_pandas_demo.ipynb)

# 01b · Excel → pandas, side by side (live demo)

We step through this together in the Day 1 pandas talk — keep it open and run each cell as it appears on screen. It shows the **two Excel moves analysts use most** — the pivot table and the VLOOKUP — built the Excel way (described) and the pandas way (run live), *including the ways each one fails*.

We use a small, readable dataset: the five-position book from Lab B, plus a reference table — small enough to check every number by eye. That is the point: learn the move where you can verify it, then apply it to 236,549 rows in Lab C.

In [ ]:
import pandas as pd
import numpy as np

trades = pd.DataFrame({
    "ticker":  ["HLT", "HLT", "MAR", "MAR", "H",  "CHH", "WH", "WH", "WH"],
    "desk":    ["Lodging", "Lodging", "Lodging", "Lodging", "Lodging", "Franchise", "Franchise", "Franchise", "Franchise"],
    "side":    ["BUY", "SELL", "BUY", "BUY", "BUY", "BUY", "BUY", "SELL", "BUY"],
    "shares":  [40_000, 10_000, 25_000, 15_000, 32_000, 28_000, 51_000, 11_000, 20_000],
    "price":   [61.40, 62.10, 155.20, 156.00, 98.75, 118.30, 72.15, 73.00, 72.60],
})
trades["notional"] = trades.shares * trades.price
trades

## 1 · The pivot table

**In Excel:** select the data → Insert → PivotTable → drag `ticker` to Rows, `side` to Columns, `notional` to Values → change "Count" to "Sum" (everyone forgets this once).

**In pandas:** one line, and it is *text* — you can read it in six months, diff it, and re-run it.

In [ ]:
pivot = trades.pivot_table(index="ticker", columns="side", values="notional", aggfunc="sum", fill_value=0)
pivot

In [ ]:
# groupby is the more general tool — most 'pivots' in practice are groupbys:
summary = trades.groupby("ticker").agg(
    trades_n=("ticker", "count"),
    total_shares=("shares", "sum"),
    gross_notional=("notional", "sum"),
    avg_price=("price", "mean"),
)
summary

In [ ]:
# Multi-level pivots — Rows: desk AND ticker. In Excel: drag two fields. Here: a list.
trades.groupby(["desk", "ticker"]).notional.sum()

## 2 · The VLOOKUP

Attach each ticker's sector and daily vol from a reference table.

**In Excel:** `=XLOOKUP(A2, ref!A:A, ref!B:B)` (or the classic `=VLOOKUP(A2, ref!A:C, 2, FALSE)` — exact match flag, column counted by hand).

**In pandas:** `merge` — a left join.

In [ ]:
ref = pd.DataFrame({
    "ticker": ["HLT", "MAR", "H", "CHH"],              # note: WH is missing — deliberately
    "sector": ["Lodging", "Lodging", "Lodging", "Franchise"],
    "daily_vol": [0.018, 0.016, 0.022, 0.019],
})
looked_up = summary.reset_index().merge(ref, on="ticker", how="left")
looked_up

### Failure case 1 — the key is missing
Excel shows `#N/A`. pandas shows `NaN`. Same disease, different rash. **Always count your NaNs after a merge:**

In [ ]:
missing = looked_up[looked_up.sector.isna()]
print(f"{len(missing)} ticker(s) missing from the reference table: {list(missing.ticker)}")
# Fix: add the row to ref (best), or .fillna() with an explicit default (document why).

### Failure case 2 — the key is duplicated on the right side
The quiet killer. **Excel's VLOOKUP silently returns the *first* match** — no warning, wrong numbers forever. pandas does something louder: it **multiplies the rows**. Louder is better: you can test for it.

In [ ]:
ref_dup = pd.concat([ref, pd.DataFrame({"ticker": ["HLT"], "sector": ["Gaming"], "daily_vol": [0.031]})])
oops = summary.reset_index().merge(ref_dup, on="ticker", how="left")
print(f"5 tickers in, {len(oops)} rows out — HLT now appears twice:")
oops[oops.ticker == "HLT"]

In [ ]:
# The guard rails, in order of preference:
# 1. validate= makes the assumption explicit — it RAISES if violated:
try:
    summary.reset_index().merge(ref_dup, on="ticker", how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:100], "...")
# 2. de-duplicate the reference table first, consciously:
ref_fixed = ref_dup.drop_duplicates(subset="ticker", keep="first")
print("after drop_duplicates:", len(summary.reset_index().merge(ref_fixed, on="ticker", how="left")), "rows — correct")

## 3 · The harder rows of the translation table

These are the "gray rows" from the slide — each worth seeing once, slowly.

In [ ]:
# IF / nested IF  →  np.where / np.select
looked_up["risk_bucket"] = np.where(looked_up.daily_vol > 0.02, "high",
                           np.where(looked_up.daily_vol > 0.017, "medium", "low"))
looked_up[["ticker", "daily_vol", "risk_bucket"]]

In [ ]:
# TRIM / PROPER / SUBSTITUTE chains  →  .str methods, chained
messy = pd.Series(["  hilton WORLDWIDE ", "marriott intl.", "HYATT hotels  "])
print(messy.str.strip().str.title().str.replace(".", "", regex=False).tolist())

In [ ]:
# VLOOKUP on TWO keys (the Excel helper-column trick)  →  merge on a list
px = pd.DataFrame({"ticker": ["HLT", "HLT", "MAR"], "date": ["2026-10-01", "2026-10-02", "2026-10-01"],
                   "close": [61.40, 62.05, 155.20]})
pos = pd.DataFrame({"ticker": ["HLT", "MAR"], "date": ["2026-10-02", "2026-10-01"], "shares": [40_000, 25_000]})
pos.merge(px, on=["ticker", "date"], how="left")        # no helper column, no CONCAT key

## The takeaways (say them back)
1. A pivot is a `groupby`; the spec is *readable text*, not a drag-and-drop you can't audit.
2. A VLOOKUP is a `merge`; count NaNs after every merge; `validate=` turns a silent Excel bug into a loud Python error.
3. The gray rows are the same ideas with more syntax — the cheat sheet (Appendix B of the design doc) has all of them.